# E2 · Regression metrics computed wrongly: four bugs

**Type**: Find the bug · **Level**: ●●○ independent · **Time**: about 30 minutes · **core path ★**

## Objective

Fix an evaluation function that runs without errors and prints plausible numbers, but computes the metrics the
wrong way, and rewrite it as a reusable function.

## Dataset

`ds.house_prices()`: 400 houses with `area`, `rooms`, `distance_center_km` and `price` in euros (skewed
distribution). The colleague trained the regression on the **logarithm** of the price, and the split is already done.

## Educational purpose

A wrong model gets noticed, a wrong metric does not. The kind of bug here is the kind that passes a review at a
glance: arguments in the wrong order, wrong units of measure, RMSE without the root, baseline computed with the
test data.

## Task

1. Run the colleague's cell. There are no errors, only numbers.
2. The true target to evaluate is the **price in euros** of the test set. Fix the computation so that `r2`, `rmse`, `rmse_baseline` and `improvement_pct` are all in euros, with the baseline predicting the mean of the **training set**.
3. Rewrite everything in `evaluate(y_true, y_pred, y_train)`, which returns a dictionary with the keys `r2`, `rmse`, `rmse_baseline`, `improvement_pct` (as a percentage, on the RMSE).
4. Use `evaluate` for two models: `log_metrics` (the colleague's model, with the predictions brought back to euros) and `direct_metrics` (a `LinearRegression` trained directly on `y_tr`). Assign to `best_by_rmse` the string `"log"` or `"direct"`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

df = ds.house_prices()
X = df[["area", "rooms", "distance_center_km"]]
y = df["price"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)

# the colleague trained the model on the logarithm of the price: there is no mistake here
model = LinearRegression().fit(X_tr, np.log(y_tr))

## Your work

### The colleague's evaluation

In [ ]:
pred_log = model.predict(X_te)
y_te_log = np.log(y_te)

r2 = r2_score(pred_log, y_te_log)
rmse = mean_squared_error(y_te_log, pred_log)
rmse_baseline = mean_squared_error(y_te_log, np.full(len(y_te), y_te_log.mean()))
improvement_pct = 100 * (1 - rmse / rmse_baseline)

print(f"R2 {r2:.3f} | RMSE {rmse:.4f} | baseline RMSE {rmse_baseline:.4f} | improvement {improvement_pct:.1f}%")

### Your version

In [ ]:
def evaluate(y_true, y_pred, y_train):
    ...
    return {"r2": ..., "rmse": ..., "rmse_baseline": ..., "improvement_pct": ...}

In [ ]:
log_metrics = ...
direct = LinearRegression().fit(X_tr, y_tr)
direct_metrics = ...
best_by_rmse = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint="The four bugs are in the colleague's cell alone: (1) the order of the arguments of `r2_score`; (2) the predictions and the target are on a logarithmic scale; (3) `mean_squared_error` returns the MSE, not the RMSE (in scikit-learn 1.1: `squared=False`); (4) the baseline uses `y_te_log.mean()`, that is information from the test set. Remember `np.exp` to go back to euros."):
    check(close(r2, 0.92547, rtol=1e-4), "correct r2 (in euros, arguments in the right order)",
          "wrong r2: is it computed on the prices in euros? r2_score wants (y_true, y_pred) in this order")
    check(close(rmse, 32853.88, rtol=1e-4), "correct rmse (in euros, with the root)",
          "wrong rmse: it must be the root of the MSE, computed on the prices in euros")
    check(close(rmse_baseline, 120444.62, rtol=1e-4), "correct baseline (mean of the training set, in euros)",
          "wrong rmse_baseline: the baseline predicts the mean of y_tr, not that of the test set")
    check(close(improvement_pct, 72.72, rtol=1e-3), "correct improvement_pct", "improvement_pct = 100 * (1 - rmse / rmse_baseline)")
    check(set(log_metrics) == {"r2", "rmse", "rmse_baseline", "improvement_pct"}, "evaluate returns the four keys",
          "the dictionary must have r2, rmse, rmse_baseline, improvement_pct")
    check(close(log_metrics["rmse"], rmse), "evaluate reproduces the correct computation for the log model",
          "evaluate(y_te, np.exp(pred_log), y_tr) should give the same rmse")
    check(best_by_rmse == ("log" if log_metrics["rmse"] < direct_metrics["rmse"] else "direct"),
          "choice consistent with the numbers", "compare log_metrics['rmse'] and direct_metrics['rmse']")
done()

---
The commented solution is in `solutions/E_find_the_bug/E2_metrics_computed_wrongly.ipynb`. Open it after you pass the check, or after you have really tried.